<a href="https://colab.research.google.com/github/sukrit37/MLPR/blob/main/YOLOv8CustomObjectDetection(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## How to Train YOLOv8 Object Detection on a Custom Dataset

In [1]:
!nvidia-smi

Fri May  8 15:21:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.6 MB/s eta 0:00:00


## Install YOLOv8

In [3]:
from ultralytics import YOLO
import os
from IPython.display import display, Image
from IPython import display
display.clear_output()
!yolo mode=checks

Traceback (most recent call last):
  File "/usr/local/bin/yolo", line 8, in <module>
    sys.exit(entrypoint())
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/__init__.py", line 943, in entrypoint
    raise ValueError(f"Invalid 'mode={mode}'. Valid modes are {list(MODES)}.\n{CLI_HELP_MSG}")
ValueError: Invalid 'mode=checks'. Valid modes are ['predict', 'train', 'val', 'benchmark', 'track', 'export'].

    Arguments received: ['yolo', 'mode=checks']. Ultralytics 'yolo' commands use the following syntax:

        yolo TASK MODE ARGS

        Where   TASK (optional) is one of ['classify', 'obb', 'detect', 'segment', 'pose']
                MODE (required) is one of ['predict', 'train', 'val', 'benchmark', 'track', 'export']
                ARGS (optional) are any number of custom 'arg=value' pairs like 'imgsz=320' that override defaults.
                    See all ARGS at https://docs.ultralytics.com/usage/cfg or with 'yolo cfg'

    1. Train a 

## Inference Example with Pretrained YOLOv8 Model

In [4]:
import os
import shutil
import glob
import yaml
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# --- Your Updated Paths ---
images_source_dir = "/content/drive/MyDrive/images"
annotations_zip = "/content/drive/MyDrive/MLPR_Datasets/my_dataset.zip"

# Working directories in Colab's temporary storage
annotations_unzip_dir = "/content/annotations_unzipped"
subset_dir = "/content/yolo_175_dataset"

# 2. Extract the annotations zip
print("Unzipping annotations...")
!unzip -o -q "{annotations_zip}" -d "{annotations_unzip_dir}"

# 3. Create the YOLOv8 required directory structure
for split in ['train', 'val']:
    os.makedirs(f"{subset_dir}/images/{split}", exist_ok=True)
    os.makedirs(f"{subset_dir}/labels/{split}", exist_ok=True)

# 4. Find and sort all images in your Drive folder to get the first 175
# (Checking multiple common extensions just in case)
all_images = glob.glob(f"{images_source_dir}/*.png") + \
             glob.glob(f"{images_source_dir}/*.jpg") + \
             glob.glob(f"{images_source_dir}/*.PNG") + \
             glob.glob(f"{images_source_dir}/*.JPG")

all_images = sorted(all_images)
first_175 = all_images[:175]

if not first_175:
    print(f"ERROR: No images found in {images_source_dir}! Please check the path.")
else:
    print(f"Found {len(all_images)} total images. Processing the first 175...")

# 5. Copy the 175 images and hunt down their matching .txt labels
count = 0
for img_path in first_175:
    base_name = os.path.basename(img_path)
    name_no_ext = os.path.splitext(base_name)[0]

    # CVAT YOLO exports usually put the text files in an 'obj_train_data' folder
    # This searches everywhere in the unzipped folder for the matching filename
    label_search = glob.glob(f"{annotations_unzip_dir}/**/{name_no_ext}.txt", recursive=True)

    # Copy image to train and val
    shutil.copy(img_path, f"{subset_dir}/images/train/{base_name}")
    shutil.copy(img_path, f"{subset_dir}/images/val/{base_name}")

    if label_search:
        # Take the matched label file and copy it
        label_path = label_search[0]
        shutil.copy(label_path, f"{subset_dir}/labels/train/{name_no_ext}.txt")
        shutil.copy(label_path, f"{subset_dir}/labels/val/{name_no_ext}.txt")
    else:
        # If no annotation exists for this frame, YOLO needs an empty txt file (background image)
        open(f"{subset_dir}/labels/train/{name_no_ext}.txt", 'w').close()
        open(f"{subset_dir}/labels/val/{name_no_ext}.txt", 'w').close()

    count += 1

# 6. Dynamically build data.yaml using CVAT's obj.names file
obj_names_path = glob.glob(f"{annotations_unzip_dir}/**/obj.names", recursive=True)

if obj_names_path:
    with open(obj_names_path[0], 'r') as f:
        classes = [line.strip() for line in f.readlines() if line.strip()]
else:
    # Fallback based on your known classes if obj.names is missing
    classes = ['car', 'person', 'license-plate', 'motorbike']

yaml_content = {
    'train': f"{subset_dir}/images/train",
    'val': f"{subset_dir}/images/val",
    'nc': len(classes),
    'names': classes
}

with open(f"{subset_dir}/data.yaml", 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"Success! Mapped {count} images and annotations. Your dataset is ready at: {subset_dir}")

Mounted at /content/drive
Unzipping annotations...
Found 1965 total images. Processing the first 175...
Success! Mapped 175 images and annotations. Your dataset is ready at: /content/yolo_175_dataset


## Train YOLOv8 Model on Custom Dataset

In [5]:
!yolo task=detect mode=train model=yolov8n.pt data=/content/yolo_175_dataset/data.yaml epochs=25 imgsz=640

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_175_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patienc

In [7]:
from google.colab import files
files.download('/content/runs/detect/train/weights/best.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
import os
import glob
from ultralytics import YOLO

# 1. Grab the next 100 images (Indices 175 to 274) from your Drive
images_source_dir = "/content/drive/MyDrive/images"

all_images = sorted(glob.glob(f"{images_source_dir}/*.png") +
                    glob.glob(f"{images_source_dir}/*.jpg") +
                    glob.glob(f"{images_source_dir}/*.PNG") +
                    glob.glob(f"{images_source_dir}/*.JPG"))

next_100_images = all_images[175:275]

# 2. Load your custom trained model
model_path = "/content/runs/detect/train/weights/best.pt"
model = YOLO(model_path)

# 3. Setup output directory
output_dir = "/content/next_100_inferences"

# 4. Run prediction
# save=True automatically saves the images with boxes drawn on them
# save_txt=True saves the bounding box coordinates in YOLO .txt format
print(f"Running inference on {len(next_100_images)} images...")
results = model.predict(
    source=next_100_images,
    conf=0.25,             # Only keep predictions with 25%+ confidence
    save=True,
    save_txt=True,
    project=output_dir,
    name="predictions"
)

print(f"Done! Your images and .txt annotations are saved in: {output_dir}/predictions")

Running inference on 100 images...

0: 480x640 2 cars, 2 persons, 3 motorbikes, 1 license-plate, 3.0ms
1: 480x640 1 car, 3.0ms
2: 480x640 1 car, 1 license-plate, 3.0ms
3: 480x640 1 person, 1 motorbike, 3.0ms
4: 480x640 1 car, 3.0ms
5: 480x640 3 cars, 2 persons, 4 motorbikes, 1 license-plate, 3.0ms
6: 480x640 1 car, 2 motorbikes, 2 license-plates, 3.0ms
7: 480x640 1 car, 3.0ms
8: 480x640 1 helmet, 1 car, 1 person, 1 motorbike, 1 license-plate, 3.0ms
9: 480x640 3 cars, 1 person, 1 motorbike, 1 license-plate, 3.0ms
10: 480x640 1 car, 2 persons, 1 motorbike, 3.0ms
11: 480x640 1 car, 3.0ms
12: 480x640 1 car, 1 license-plate, 3.0ms
13: 480x640 2 cars, 1 license-plate, 3.0ms
14: 480x640 1 helmet, 1 car, 1 person, 1 motorbike, 1 license-plate, 3.0ms
15: 480x640 1 car, 1 motorbike, 3.0ms
16: 480x640 1 car, 1 license-plate, 3.0ms
17: 480x640 1 car, 3.0ms
18: 480x640 1 car, 3.0ms
19: 480x640 2 cars, 1 license-plate, 3.0ms
20: 480x640 2 cars, 2 license-plates, 3.0ms
21: 480x640 1 car, 1 license-pl

In [12]:
import cv2
import os
import glob

# ==========================================
# 1. Setup Your Paths
# ==========================================
raw_images_dir = "/content/drive/MyDrive/images"
labels_dir = "/content/next_100_inferences/predictions/labels"
output_viz_folder = "/content/custom_visual_checks"

os.makedirs(output_viz_folder, exist_ok=True)

# ==========================================
# 2. Correct Class Mapping & Colors
# ==========================================
# Extracted directly from your CVAT annotations.xml
class_names = {
    0: 'turban',
    1: 'helmet',
    2: 'car',
    3: 'person',
    4: 'motorbike',
    5: 'license-plate',
    6: 'autorickshaw',
    7: 'animal',
    8: 'truck',
    9: 'bus',
    10: 'unknown',
    11: 'no-helmet'
}

# Assigning unique colors for visual distinction (OpenCV uses B, G, R format)
label_colors = {
    'turban': (0, 165, 255),        # Orange
    'helmet': (255, 0, 0),          # Blue
    'car': (0, 255, 0),             # Green
    'person': (0, 0, 255),          # Red
    'motorbike': (255, 255, 0),     # Cyan
    'license-plate': (255, 0, 255), # Magenta
    'autorickshaw': (0, 255, 255),  # Yellow
    'animal': (128, 0, 128),        # Purple
    'truck': (128, 128, 0),         # Teal
    'bus': (0, 128, 128),           # Olive
    'unknown': (128, 128, 128),     # Gray
    'no-helmet': (0, 0, 0),         # Black
    'default': (255, 255, 255)      # White fallback
}

# ==========================================
# 3. Draw Bounding Boxes on 20 Images
# ==========================================
# Grab the first 20 .txt prediction files
txt_files = sorted(glob.glob(f"{labels_dir}/*.txt"))[:20]

processed_count = 0

for i, txt_path in enumerate(txt_files):
    # The prediction labels are saved as image0.txt, image1.txt, etc.
    # We need to use the original image path based on its index
    if i < len(next_100_images):
        img_path = next_100_images[i]
    else:
        print(f"Warning: No corresponding original image found for {txt_path}")
        continue

    # Read the image
    image = cv2.imread(img_path)
    if image is None:
        print(f"Warning: Could not read image {img_path}")
        continue

    img_height, img_width, _ = image.shape

    # Parse the YOLO text file
    with open(txt_path, 'r') as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()
        if len(parts) < 5:
            continue

        class_id = int(parts[0])
        x_center = float(parts[1])
        y_center = float(parts[2])
        box_width = float(parts[3])
        box_height = float(parts[4])

        # Convert YOLO normalized coordinates to absolute pixel coordinates
        x_center_px = x_center * img_width
        y_center_px = y_center * img_height
        w_px = box_width * img_width
        h_px = box_height * img_height

        xtl = int(x_center_px - (w_px / 2))
        ytl = int(y_center_px - (h_px / 2))
        xbr = int(x_center_px + (w_px / 2))
        ybr = int(y_center_px + (h_px / 2))

        # Get label string and color
        label = class_names.get(class_id, "unknown_class")
        color = label_colors.get(label, label_colors['default'])

        # Draw the rectangle
        cv2.rectangle(image, (xtl, ytl), (xbr, ybr), color, 2)

        # Draw label background box for better text readability
        (text_width, text_height), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(image, (xtl, ytl - text_height - 5), (xtl + text_width, ytl), color, -1)

        # Add text label in white (or black depending on background)
        text_color = (255, 255, 255) if label != 'no-helmet' else (0, 255, 255)
        cv2.putText(image, label, (xtl, ytl - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, text_color, 1)

    # Save output
    output_path = os.path.join(output_viz_folder, os.path.basename(img_path))
    cv2.imwrite(output_path, image)
    processed_count += 1

print(f"Visualization complete!")
print(f"Processed and saved {processed_count} images to: {output_viz_folder}")

Visualization complete!
Processed and saved 0 images to: /content/custom_visual_checks
